To [Film Pit](https://thefilmpit.com) είναι ένα podcast το οποίο διανύει τον τέταρτο χρόνο προβολής του. Ασχολείται με την κριτική ταινιών μικρής δημοφιλίας και προϋπολογισμού κυρίως από την δεκαετία του 80 και 90. Στο διάστημα αυτό οι 3 δημιουργοί του έχουν παρουσιάσει και σχολιάσει 114 ταινίες. Όντας ακροατής, θέλησα να βρω περισσότερες πληροφορίες για τις ταινίες που επέλεξαν.

Παρακάτω σας παρουσιάζω τον κώδικα Python που χρησιμοποιήθηκε για την συγκέντρωση των πληροφοριών από τις ιστοσελίδες Internet Movie Database (IMDB) και The Movie Database (TMDB). Οι βιβλιοθήκες που χρησιμοποιήθηκαν είναι οι `requests`, `BeautifulSoup`, `pandas` και `numpy`.

Οι πληροφορίες παρουσιάζονται υπό μορφή γραφημάτων με την βοήθεια του προγράμματος Tableau. Αν θέλετε να δείτε απευθείας το τελικό αποτέλεσμα μπορείτε να το βρείτε στην @sec-tableau της σελίδας, είτε απευθείας στο [Tableau public.](https://public.tableau.com/views/Filmpit/FilmpitMovies?:language=en-US&:display_count=n&:origin=viz_share_link)


## Συγκέντρωση των τίτλων

Αρχικά με τις βιβλιοθήκες `requests` και `BeautifulSoup` συγκέντρωσα όλους τους τίτλους των ταινιών από την ιστοσελίδα του Film Pit και τις αποθήκευσα σε ένα αρχείο με όνομα `movie_titles.csv`.

In [1]:
# Φόρτωση των απαραίτητων βιβλιοθηκών
import requests
from bs4 import BeautifulSoup
import time
import pandas as pd
import numpy as np
from os import path

In [2]:
# Ελέγχουμε έαν υπάρχει ήδη το αρχείο αλλιώς το δημιουργούμε
if path.exists(path.join('datasets','movie_titles.csv')):
    movie_titles = pd.read_csv(path.join('datasets', 'movie_titles.csv')).squeeze('columns')
else:
    URL = "https://thefilmpit.com"

    def get_movies_title(URL):
        """Scrapes podcasts hrefs to get movie titles"""
        movie_titles = []
        page = requests.get(URL)
        next_link = None   
        
        if page.ok:
            soup = BeautifulSoup(page.content, 'html.parser')
            try:
                # Ελέγχουμε αν υπάρχει επόμενη σελίδα στο site
                next_link = soup.find('link', {'rel':'next',}).get('href')
            except AttributeError:
                print("This page hasn't a next link")

            # Απο τους συνδέσμους href λαμβάνουμε τους τίτλους
            for podcast in soup.find_all('a', attrs={'rel':'bookmark'}): 
                podcast_link = podcast.get('href')
                title = podcast_link.split('/')[-2].replace('-', ' ')
                movie_titles.append(title)
        return movie_titles, next_link

    next_link = URL
    movie_titles = []

    while next_link:
        print(f"Scraping next link: {next_link}")
        titles, next_link = get_movies_title(next_link)
        movie_titles += titles
        time.sleep(5)
    print("Scraping finished.")

In [3]:
# Κάποιοι τίτλοι χρειάζονται διόρθωση
if not path.exists(path.join('datasets','movie_titles.csv')):
    movie_titles += ['never too young to die', 'Dr Caligari', 'Yeti'] # 3 ταινίες δεν υπάρχουν στην ιστοσελίδα
    movie_titles = pd.Series(movie_titles)
    
    fixed_titles = {'diagalaxiaki poiotita galaxy of terror': 'galaxy of terror',
                    'i scholi tou gkontfrei sakura killers': 'sakura killers',
                    'exairetika petsino podkast the punisher': 'the punisher',
                    'o rambu tis indonisias einai edo': 'Rambu aka The Intruder',
                    'tha einai san star gouorz alla den tha einai star gouorz battle beyond the stars': 'battle beyond the stars',
                    'brady idrotas kai pioti tie night stalker': 'night stalker',
                    'pao na kano penintarika double dragon': 'double dragon',
                    'mousikorama shock em dead': 'shock em dead',
                    'asiatiki tourne 3 undefeatable': 'undefeatable',
                    'asiatiki tourne 2 w is war': 'w is war',
                    'asiatiki tourne 1 for your height only': 'for your height only',
                    'ena mikro mousiko breik rappin': 'rappin',
                    'dark night of the scarecrow feat elina dimitriadi': 'dark night of the scarecrow',
                    'prom night feat elina dimitriadi': 'prom night',
                    'aerobicide': 'Killer Workout',
                    'an american hippie in paris': 'An American Hippie in Israel',
                    'tc2000': 'tc 2000',
                    'american ninia': 'american ninja',
                    'class of nukem high': "Class of Nuke 'Em High",
                    'superman 4': "superman iv"
                    }

    movie_titles.replace(fixed_titles, inplace=True)
    movie_titles.drop([91, 109], inplace=True)

In [4]:
# Αποθήκευση των τίτλων στο αρχείο movie_title.csv

if not path.exists(path.join('datasets','movie_titles.csv')):
    movie_titles.to_csv(path.join('datasets','movie_titles.csv'), index=False, header=['titles'])

## Εξαγωγή πληροφοριών από την ιστοσελίδα The Movie Database

Για την εξαγωγή πληροφοριών από το [TMDB](https://www.themoviedb.org) χρησιμοποίησα την βιβλιοθήκη `tmdbv3api`.

In [5]:
if path.exists(path.join('datasets','movies_tmdb.csv')):
    movies_tmdb = pd.read_csv(path.join('datasets','movies_tmdb.csv'), index_col=0)
else:
    from tmdbv3api import TMDb
    from config import config
    tmdb = TMDb()
    tmdb.api_key = config['tmdb_api_key'] # το api key δίνεται δωρεάν με απλή εγγραφή στο TMDB

    from tmdbv3api import Movie
    movie = Movie()

    basic_info = {}
    not_found = []
    for title in movie_titles.to_list():
        print('Fetching ' + title)
        try:
            # Χρησιμοποιούμε το πρώτο αποτέλεσμα αναζήτησης
            basic_info[title] = movie.search(title)[0] 
        except IndexError:
            # Αποθηκεύουμε τους τίτλους που δεν βρέθηκαν στην λίστα not_found
            not_found.append(title) 

    print('Movies not found: ', not_found)

In [6]:
# Ελέγχουμε ότι το πρώτο αποτέλεσμα αναζήτησης ήταν το σωστό
if not path.exists(path.join('datasets','movies_tmdb.csv')):
    for title in movie_titles.to_list():
        if basic_info.get(title):
            print(title, ': ', basic_info[title]['title'], basic_info[title]['release_date'][:4])

Δυστυχώς τα πρώτο αποτέλεσμα από κάθε ταινία δεν είναι πάντα το επιθυμητό δεδομένου ότι πολλές ταινίες έχουν τον ίδιο ή παρόμοιο τίτλο. Παρακάτω επιλέγουμε τις σωστές ταινίες μέσω του αριθμού `id` κάθε ταινίας.

In [7]:
if not path.exists(path.join('datasets','movies_tmdb.csv')):
    
    
    wrong_matches = {'Rambu aka The Intruder': '81944',
                     'lambada set the night on fire': '117269',
                     'the baby': '28156',
                     'conquest': '27232',
                     'suburbia': '28054',
                     'warlock': '11342',
                     'to kako': '39897',
                     'commander': '205697',
                     'captain america': '13995',
                     'star wars holiday special': '74849',
                     'the punisher': '8867',
                     'night stalker':'66474',
                     'the perfect weapon': '34421',
                     'the rage': '114936',
                     'warbirds': '219359',
                     'arena': '44796',
                     'jack frost': '27318',
                     'elves': '30452',
                     'prom night': '36599',
                     'cheerleader camp': '40087',
                     'endgame': '28850',
                     'thunder': '109104',
                     'Dr Caligari': '35642',
                     'Yeti': '92316'}
    

    movie_ids = {title:info['id'] for title, info in basic_info.items()}
    movie_ids.update(wrong_matches)

Εφόσον πλέον έχουμε το σωστό `id` για όλες τις ταινίες εξάγουμε όλες τις πληροφορίες που χρειαζόμαστε, τις αποθηκεύουμε στο λεξικό `movie_records` και με την βιβλιοθήκε `pandas` δημιουργούμε το αρχείο `movies_tmdb.csv`.

In [8]:
if not path.exists(path.join('datasets','movies_tmdb.csv')):
    print('Starting api request')
    movie_records = {}
    for title, id in movie_ids.items():
        
        print(f"Fetching {title}")
        mov = movie.details(id)
        temp_list = [
            mov.imdb_id,
            mov.original_title,
            mov.budget,
            mov.revenue,
            mov.runtime,
            mov.popularity,
            [company['name'] for company in mov.production_companies],
            [key['name'] for key in mov.keywords.keywords],
            [act['name'] for act in mov.casts.cast],
            mov.overview
        ]
        movie_records[mov.title] = temp_list
    
    print('TMDB request finished')
    columns = ['imdb_id',
               'original_title',
               'budget',
               'revenue',
               'runtime',
               'popularity',
               'production_companies',
               'keywords',
               'cast',
               'overview'
              ]
    movies_tmdb = pd.DataFrame.from_dict(movie_records, orient='index', columns=columns)

In [9]:
# Αντικαθιστούμε το 0 ως τιμή που λείπει
movies_tmdb[['budget', 'revenue', 'runtime']] = movies_tmdb[['budget', 'revenue', 'runtime']].replace({0: np.nan})

Παρακάτω είναι ένα μικρό δείγμα του αρχείου.

In [10]:
movies_tmdb.sample(5)

,imdb_id,original_title,budget,revenue,runtime,popularity,overview
Superman IV: The Quest for Peace,tt0094074,Superman IV: The Quest for Peace,17000000.0,19300000.0,90.0,25.238,With global superpowers engaged in an increasi...
2019: After the Fall of New York,tt0085125,2019 - Dopo la caduta di New York,NaN,NaN,96.0,4.987,After a nuclear war society breaks down into t...
Evil,tt0813129,Το Κακό,NaN,NaN,83.0,0.877,An evil force is awakened in downtown Athens t...
SS Experiment Love Camp,tt0074768,Lager SSadis Kastrat Kommandantur,NaN,NaN,94.0,4.011,"Near the end of WW2, prisoners of war are used..."
Warbirds,tt0096417,Warbirds,NaN,NaN,NaN,0.710,A covert military airstrike against rebels in ...


Κάθε ταινία έχει πολλαπλές τιμές σε κάθε πεδίο. Για την επεξεργασία όλων των πληροφοριών θα πρέπει ή να δημιουργήσουμε μία βάση sql με πολλαπλούς πίνακες ή να δημιουργήσουμε διαφορετικά αρχεία csv που θα μπορούν να συνδυαστούν μεσω του κοινού πεδίου `imdb_id`. Παρακάτω παρουσιάζεται η δεύτερη μέθοδος. Τα αρχεία θα συνδυαστούν αργότερα στο πρόγραμμα Tableau μέσω των data relationships.

In [11]:
# Αρχείο companies.csv που θα περιέχει τις πληροφορίες των εταιριών παραγωγής
if not path.exists(path.join('datasets','companies.csv')):
    companies = (movies_tmdb[['imdb_id', 'production_companies']]
                 .reset_index()
                 .rename(columns={'index':'title'})
                 .explode(column='production_companies')
                 .dropna().reset_index(drop=True)).copy()
    companies.to_csv(path.join('datasets','companies.csv'), index=False)

In [12]:
# Αρχείο cast.csv που θα περιέχει τους ηθοποιούς ανά ταινία
if not path.exists(path.join('datasets','cast.csv')):
    cast = (movies_tmdb[['imdb_id', 'cast']]
             .reset_index()
             .rename(columns={'index':'title'})
             .explode(column='cast')
             .dropna().reset_index(drop=True)).copy()
    cast.to_csv(path.join('datasets','cast.csv'), index=False)

In [13]:
# Αρχείο keywords.csv που θα περιέχει τις λέξεις κλειδιά για την περιγραφή κάθε ταινίας
if not path.exists(path.join('datasets','keywords.csv')):
    keywords = (movies_tmdb[['imdb_id', 'keywords']]
                .reset_index().rename(columns={'index':'title'})
                .explode(column='keywords')
                .dropna().reset_index(drop=True).copy())
    keywords.to_csv(path.join('datasets', 'keywords.csv'), index=False)

In [15]:
# Οι υπόλοιπες στήλες θα αποθηκευτούν σε ένα αρχείο με όνομα movies_tmdb.csv
print("Στήλες: ", movies_tmdb.columns.values)

Στήλες:  ['imdb_id' 'original_title' 'budget' 'revenue' 'runtime' 'popularity'
 'overview']


In [16]:
if not path.exists(path.join('datasets','movies_tmdb.csv')):
    movies_tmdb.drop(columns=['production_companies', 'keywords', 'cast'], inplace=True)
    movies_tmdb.to_csv(path.join('datasets','movies_tmdb.csv'))

## Εξαγωγή πληροφοριών από την ιστοσελίδα Open Movie Database

Αρχικά προσπάθησα να χρησιμοποιήσω την βιβλιοθήκη `omdb` αλλά τελικά είχα καλύτερα αποτελέσματα με το api της ιστοσελίδας [OMDb](https://www.omdbapi.com). Η OMDb παρέχει τις ίδιες πληροφορίες που έχει το IMDB αλλά με ευκολότερη πρόσβαση. Οι πληροφορίες θα αποθηκευτούν στο λεξικό `omdb_info` και μετέπειτα σε αρχείο `movies_omdb.csv` με την βοήθεια του `pandas`.

In [17]:
if path.exists(path.join('datasets','movies_omdb.csv')):
    movies_omdb = pd.read_csv(path.join('datasets','movies_omdb.csv'), index_col=0)
else:
    from config import config # αποθήκευση των κλειδιών api στο βοηθητικό αρχείο 'config.py'
    omdb_info = {}
    for imdb_id in movies_tmdb['imdb_id'].to_list():
        try:
            res = requests.get(f"http://www.omdbapi.com/?i={imdb_id}&apikey={config['omdb_api_key']}", timeout=3).json()
            omdb_info[imdb_id] = [
                res['Year'],
                res['Rated'],
                res['Genre'],
                res['Director'],
                res['Writer'],
                res['Language'],
                res['Country'],
                res['Awards'],
                res['Metascore'],
                res['imdbRating'],
                res['imdbVotes'],
                [rating['Value'] for rating in res['Ratings'] if rating['Source'] == 'Rotten Tomatoes']
            ]
        except Exception:
            print(f"Request for movie id {imdb_id} did not executed.")

In [18]:
if not path.exists(path.join('datasets','movies_omdb.csv')):
    columns = ['year',
               'rated',
               'genre',
               'director',
               'writer',
               'language',
               'country',
               'awards',
               'metascore',
               'imdb_rating',
               'imdb_votes',
               'rotten_rating']
    movies_omdb = pd.DataFrame.from_dict(omdb_info, orient='index', columns=columns)
    movies_omdb = movies_omdb.reset_index().rename(columns={'index': 'imdb_id'})
    movies_omdb.replace({'N/A': np.nan}, inplace=True)
    
    # Βελτίωση των αποτελεσμάτων των ratings
    movies_omdb['imdb_votes'] = movies_omdb['imdb_votes'].str.replace(',', '').astype(int)
    movies_omdb['rotten_rating'] = movies_omdb.rotten_rating.apply(lambda x: x[0] if x else np.nan).str.replace('%', '').astype(float)
    movies_omdb['imdb_rating'] = movies_omdb.imdb_rating.astype(float)
    movies_omdb['metascore'] = movies_omdb.metascore.astype(float)

In [20]:
movies_omdb.sample(5)

,imdb_id,year,rated,awards,metascore,imdb_rating,imdb_votes,rotten_rating
79,tt0117433,1997,R,NaN,NaN,4.4,572,NaN
85,tt0087945,1986,R,NaN,NaN,3.6,1076,NaN
56,tt0489210,2009,PG,NaN,NaN,2.1,560,NaN
30,tt0088708,1985,R,NaN,20.0,5.4,15243,0.0
58,tt0090837,1986,R,NaN,22.0,5.6,14561,50.0


In [ ]:
movies_omdb.info()

Όπως και στο TMDB υπάρχουν πολλαπλές τιμές για κάθε πεδίο της εκάστοτε ταινίας οπότε δημιούργησα μία εφαρμογή που θα αποθηκεύει αυτές τις τιμές σε ξεχωριστό αρχείο csv μαζί με ένα αναγνωριστικό για την κάθε ταινία.

In [ ]:
def create_sup_tables(column, f_name):
    """
    The fuction picks a dataframe column, splits values by coma,
    creates different columns and writes the new dataframe
    to a csv file based on f_name
    """
    df = movies_omdb[['imdb_id', column]].copy()
    df[column] = df[column].str.split(',')
    df = df.explode(column=column)
    df[column] = df[column].str.strip()
    df.dropna(inplace=True)
    df.reset_index(drop=True, inplace=True)
    df.to_csv(path.join('datasets', f_name), index=False)

In [ ]:
# Χρήση της εφαρμογής σε 5 στήλες

if not path.exists(path.join('datasets','genres.csv')):
    create_sup_tables('genre', 'genres.csv')
    
if not path.exists(path.join('datasets','writers.csv')):
    create_sup_tables('writer', 'writers.csv')
    
if not path.exists(path.join('datasets','countries.csv')):
    create_sup_tables('country', 'countries.csv')
    
if not path.exists(path.join('datasets','languages.csv')):
    create_sup_tables('language', 'languages.csv')
    
if not path.exists(path.join('datasets','directors.csv')):
    create_sup_tables('director', 'directors.csv')

Εξάγουμε τις υπόλοιπες στήλες για τις οποίες δεν δημιουργήσαμε ξεχωριστά αρχεία στο αρχείο `movies_omdb.csv`.

In [ ]:
if not path.exists(path.join('datasets','movies_omdb.csv')):
    movies_omdb.drop(columns=['genre', 'writer', 'country', 'director', 'language'], inplace=True)
    movies_omdb.to_csv(path.join('datasets','movies_omdb.csv'))

Τέλος ενώνουμε τις πληροφορίες απο τις δύο ιστοσελίδες σε ένα αρχείο που ονομάζεται `movies.csv`.

In [ ]:
movies = movies_tmdb.reset_index().rename(columns={'index':'title'}).merge(movies_omdb, how='inner', on='imdb_id')

movies.info()

if not path.exists(path.join('datasets','movies.csv')):
    movies.to_csv(path.join('datasets','movies.csv'), index=False)

## Χρήση του προγράμματος Tableau Public για την παρουσίαση των δεδομένων. {#sec-tableau}

Τα γραφήματα που δημιουργούμε μπορούν να συνδιαστούν σε dashboards και να παρουσιαστούν υπό την μορφή stories, αναρτημένα στην υπηρεσία Τableau Public. Παρακάτω βλέπεται το αποτέλεσμα.


In [21]:
#| code-fold: true
#| column: page

%%html

<div class='tableauPlaceholder' id='viz1666283627204' style='position: relative'><noscript><a href='#'><img alt='Filmpit Movies ' src='https:&#47;&#47;public.tableau.com&#47;static&#47;images&#47;Fi&#47;Filmpit&#47;FilmpitMovies&#47;1_rss.png' style='border: none' /></a></noscript><object class='tableauViz'  style='display:none;'><param name='host_url' value='https%3A%2F%2Fpublic.tableau.com%2F' /> <param name='embed_code_version' value='3' /> <param name='site_root' value='' /><param name='name' value='Filmpit&#47;FilmpitMovies' /><param name='tabs' value='no' /><param name='toolbar' value='yes' /><param name='static_image' value='https:&#47;&#47;public.tableau.com&#47;static&#47;images&#47;Fi&#47;Filmpit&#47;FilmpitMovies&#47;1.png' /> <param name='animate_transition' value='yes' /><param name='display_static_image' value='yes' /><param name='display_spinner' value='yes' /><param name='display_overlay' value='yes' /><param name='display_count' value='yes' /><param name='language' value='en-US' /></object></div>                <script type='text/javascript'>                    var divElement = document.getElementById('viz1666283627204');                    var vizElement = divElement.getElementsByTagName('object')[0];                    vizElement.style.width='1016px';vizElement.style.height='991px';                    var scriptElement = document.createElement('script');                    scriptElement.src = 'https://public.tableau.com/javascripts/api/viz_v1.js';                    vizElement.parentNode.insertBefore(scriptElement, vizElement);                </script>